# Statistică inferențială - Testarea ipotezelor de cercetare

Notebook-ul testează formal ipotezele formulate pentru RQ1–RQ4, pornind de la datasetul consolidat `df_master.csv`, generat în notebook-ul EDA.

Testele statistice sunt organizate în jurul celor patru întrebări de cercetare și țin cont de cele două axe principale ale evaluării:

1. **Timpul de execuție**, analizat doar pentru interogările finalizate cu succes (`SUCCESS`);
2. **Executabilitatea**, analizată prin statusul execuției (`SUCCESS`, `CITUS_ARCH_ERROR`, `TIMEOUT_3600`, `SERVER_CRASH`) sau prin variabila binară `success`.

Structura:

1. Setup și import date
2. RQ1 — Baseline vs scenarii distribuite
3. RQ2 — Scalare N2 vs N5
4. RQ3 — Strategie D1 vs D2
5. RQ4 — Caracteristici structurale SQL vs timp / executabilitate
6. Sinteză finală a ipotezelor

In [1]:
library(tidyverse)
library(janitor)
library(scales)
library(ggstatsplot)
library(effectsize)
library(rstatix)
library(patchwork)
library(knitr)
library(viridis)
library(ggsci)
library(svglite)

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'tidyr' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'forcats' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.5.1
✔ ggplot2   4.0.3     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to 

In [2]:
base_dir <- ".."

df_master_file <- file.path(base_dir, "citus_outputs","df_master.rds")

df_master <- readRDS(df_master_file)

glimpse(df_master)

Rows: 3,750
Columns: 52
$ scenario            <chr> "SF10_N0", "SF10_N0", "SF10_N0", "SF10_N0", "SF10_…
$ sf                  <dbl> 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10…
$ workers             <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ strategy            <chr> "baseline", "baseline", "baseline", "baseline", "b…
$ query_no            <dbl> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,…
$ query_id            <chr> "Q001", "Q002", "Q003", "Q004", "Q005", "Q006", "Q…
$ status              <chr> "SUCCESS", "SUCCESS", "SUCCESS", "SUCCESS", "SUCCE…
$ error_category      <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA, NA…
$ elapsed_ms          <dbl> 14165, 6954, 11010, 12679, 29054, 114241, 1416, 36…
$ elapsed_sec         <dbl> 14.165, 6.954, 11.010, 12.679, 29.054, 114.241, 1.…
$ success             <lgl> TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TR…
$ response_code       <chr> "200", "200", "200", "200", "200", "200", "200", "…
$ response_messa

In [3]:
tables_dir <- file.path(base_dir, "citus_outputs", "tables")
figures_dir <- file.path(base_dir, "citus_outputs", "figures")

In [8]:
df_tests <- df_master %>%
  select(
    # identificare și design experimental
    scenario, sf, workers, strategy, architecture, query_id, query_no,
    
    # rezultat execuție
    status, success, elapsed_sec,
    
    # variabile structurale numerice
    n_tables, n_joins, n_inner_joins, n_left_joins, n_right_joins,
    n_full_joins, n_cross_joins, n_subqueries, n_ctes,
    n_aggregate_types, n_aggregate_calls,
    n_where_predicates, n_having_predicates,
    
    # variabile structurale binare
    starts_with("has_")
  ) %>%
  mutate(
    log_elapsed_sec = log10(elapsed_sec),
    status = factor(status, levels = c(
      "SUCCESS",
      "CITUS_ARCH_ERROR",
      "TIMEOUT_3600",
      "SERVER_CRASH"
    )),
    strategy = factor(strategy, levels = c("baseline", "D1", "D2")),
    architecture = factor(architecture, levels = c("baseline", "distributed"))
  )

glimpse(df_tests)

Rows: 3,750
Columns: 44
$ scenario            <chr> "SF10_N0", "SF10_N0", "SF10_N0", "SF10_N0", "SF10_…
$ sf                  <dbl> 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10…
$ workers             <dbl> 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,…
$ strategy            <fct> baseline, baseline, baseline, baseline, baseline, …
$ architecture        <fct> baseline, baseline, baseline, baseline, baseline, …
$ query_id            <chr> "Q001", "Q002", "Q003", "Q004", "Q005", "Q006", "Q…
$ query_no            <dbl> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,…
$ status              <fct> SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCESS, SUCCE…
$ success             <lgl> TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TRUE, TR…
$ elapsed_sec         <dbl> 14.165, 6.954, 11.010, 12.679, 29.054, 114.241, 1.…
$ n_tables            <dbl> 3, 3, 4, 6, 4, 4, 2, 3, 3, 3, 3, 4, 4, 4, 3, 4, 3,…
$ n_joins             <dbl> 2, 2, 3, 5, 3, 3, 1, 2, 3, 3, 2, 3, 2, 3, 2, 3, 4,…
$ n_inner_joins 

Am creat dataframe-ul `df_tests`, pornind de la `df_master`, din care am păstrat variabilele relevante pentru testele inferențiale: identificatori, factori experimentali, statusul execuției, timpul de execuție și caracteristicile structurale ale interogărilor.

Variabilele categoriale principale au fost convertite în factori cu ordine explicită, iar `log_elapsed_sec` a fost calculat pentru testele și vizualizările care folosesc timpul de execuție pe scară logaritmică.

In [9]:
library(skimr)

skim(df_tests)

── Data Summary ────────────────────────
                           Values  
Name                       df_tests
Number of rows             3750    
Number of columns          44      
_______________________            
Column type frequency:             
  character                2       
  factor                   23      
  logical                  1       
  numeric                  18      
________________________           
Group variables            None    

── Variable type: character ────────────────────────────────────────────────────
  skim_variable n_missing complete_rate min max empty n_unique whitespace
1 scenario              0             1   7  11     0       15          0
2 query_id              0             1   4   4     0      250          0

── Variable type: factor ───────────────────────────────────────────────────────
   skim_variable  n_missing complete_rate ordered n_unique
 1 strategy               0             1 FALSE          3
 2 architecture        

ERROR: Error in is.null(text_repr) || nchar(text_repr) == 0L: 'length = 22' in coercion to 'logical(1)'


In [10]:
skim_result <- skim(df_tests)

write_csv(
  as.data.frame(skim_result),
  file.path(tables_dir, "df_tests_skim_summary.csv")
)

In [11]:
df_tests_file <- file.path(base_dir, "citus_outputs", "df_tests.csv")

write_csv(df_tests, df_tests_file)

df_tests_rds <- file.path(base_dir, "citus_outputs", "df_tests.rds")

saveRDS(df_tests, df_tests_rds)

In [12]:
normality_check_time <- df_tests %>%
  filter(success == TRUE) %>%
  summarise(
    shapiro_elapsed_sec_p = shapiro.test(elapsed_sec)$p.value,
    shapiro_log_elapsed_sec_p = shapiro.test(log_elapsed_sec)$p.value
  )

normality_check_time

shapiro_elapsed_sec_p,shapiro_log_elapsed_sec_p
<dbl>,<dbl>
1.282903e-74,2.07395e-13


**Justificarea testelor neparametrice.** 

În notebook-ul EDA, distribuțiile timpilor de execuție și ale caracteristicilor structurale numerice au fost identificate ca fiind asimetrice, cu valori extreme. Pentru variabilele structurale numerice, testele Shapiro-Wilk au indicat p-value sub pragul de 0,05, ceea ce duce la respingerea ipotezei de normalitate. În consecință, testele inferențiale din acest notebook folosesc metode neparametrice: Wilcoxon signed-rank pentru comparații pereche, Mann-Whitney/Wilcoxon rank-sum pentru grupuri independente, McNemar pentru date binare pereche și Spearman pentru corelații.

A significant result from the Shapiro-Wilk test indicates a rejection of the null hypothesis that the data is normally distributed. Therefore, if the Shapiro-Wilk test is significant, the appropriate course of action typically involves using non-parametric methods as they do not assume normality.

[p-value interpretation](https://www.atlas.org/spaces/solve/understanding-p-value-interpretation-q9C7pZkZdEJfKYnKKzddLr) 

Testele sunt organizate în patru familii, câte una pentru fiecare întrebare de cercetare. Pentru fiecare familie se aplică **corecția Holm** pentru testare multiplă, iar **mărimile efectului** sunt raportate alături de p-value, acolo unde acestea sunt aplicabile.

În cadrul analizelor sunt utilizate următoarele mărimi ale efectului:
- **rank-biserial correlation** pentru testele Wilcoxon signed-rank și Mann-Whitney U / Wilcoxon rank-sum;
- **odds ratio pe perechile discordante** pentru testele McNemar;
- **rho Spearman** pentru corelațiile dintre variabile numerice;
- pentru testele Fisher exact, interpretarea se bazează pe diferențele dintre ratele de succes și pe semnificația statistică, deoarece unele caracteristici structurale sunt rare și pot produce estimări instabile ale mărimii efectului.

Corecția Holm este aplicată separat pe fiecare familie de teste, pentru a reduce riscul de rezultate fals pozitive în contextul testării multiple.

# RQ1  Baseline vs scenarii distribuite
 
RQ1. În ce măsură o arhitectură distribuită bazată pe Citus îmbunătățește timpii de execuție ai interogărilor analitice complexe, comparativ cu un sistem PostgreSQL mononod?

H01: Nu există diferențe semnificative între timpii de execuție ai interogărilor în scenariile PostgreSQL mononod și scenariile distribuite prin Citus.

H11: Scenariile distribuite cu Citus prezintă timpi de execuție semnificativ diferiți față de scenariile PostgreSQL mononod, cu tendința de reducere a timpilor pentru anumite volume de date și clase de interogări.

 
  - Timp: 12 teste Wilcoxon signed-rank pereche (4 scenarii distribuite × 3 SF), pe interogările SUCCESS în ambele scenarii comparate, **bilateral** (H11 nondirecțional)
  - Executabilitate: 12 teste McNemar pe `success` (4 × 3 SF)

## A. Timpi de execuție -12 teste Wilcoxon signed-rank pereche

In [13]:
# Funcție helper: rulează Wilcoxon signed-rank pereche pentru o pereche de scenarii
run_paired_wilcox_time <- function(baseline_scen, distrib_scen, df) {
  paired_df <- df %>%
    filter(scenario %in% c(baseline_scen, distrib_scen)) %>%
    filter(status == "SUCCESS") %>%
    select(query_id, scenario, elapsed_sec) %>%
    pivot_wider(names_from = scenario, values_from = elapsed_sec) %>%
    drop_na()
  
  baseline_times <- paired_df[[baseline_scen]]
  distrib_times  <- paired_df[[distrib_scen]]
  
  test <- wilcox.test(baseline_times, distrib_times,
                      paired = TRUE, alternative = "two.sided",
                      conf.int = TRUE, exact = FALSE)
  
  eff <- effectsize::rank_biserial(baseline_times, distrib_times, paired = TRUE)
  
  tibble(
    n_pairs            = nrow(paired_df),
    median_baseline    = round(median(baseline_times), 3),
    median_distributed = round(median(distrib_times), 3),
    statistic_V        = unname(test$statistic),
    p_value            = test$p.value,
    rank_biserial      = round(eff$r_rank_biserial, 3),
    effect_magnitude   = effectsize::interpret_rank_biserial(eff$r_rank_biserial)
  )
}

In [14]:
# Cele 12 combinații RQ1: pentru fiecare SF, baseline vs fiecare scenariu distribuit
rq1_comparisons <- expand_grid(
  sf       = c(10, 50, 100),
  workers  = c(2, 5),
  strategy = c("D1", "D2")
) %>%
  mutate(
    baseline_scenario    = paste0("SF", sf, "_N0"),
    distributed_scenario = paste0("SF", sf, "_N", workers, "_", strategy)
  )

rq1_comparisons

sf,workers,strategy,baseline_scenario,distributed_scenario
<dbl>,<dbl>,<chr>,<chr>,<chr>
10,2,D1,SF10_N0,SF10_N2_D1
10,2,D2,SF10_N0,SF10_N2_D2
10,5,D1,SF10_N0,SF10_N5_D1
10,5,D2,SF10_N0,SF10_N5_D2
50,2,D1,SF50_N0,SF50_N2_D1
50,2,D2,SF50_N0,SF50_N2_D2
50,5,D1,SF50_N0,SF50_N5_D1
50,5,D2,SF50_N0,SF50_N5_D2
100,2,D1,SF100_N0,SF100_N2_D1


In [15]:
# cele 12 teste cu Holm pentru testarea multiplă
rq1_time_results <- rq1_comparisons %>%
  mutate(
    test_results = map2(
      baseline_scenario, distributed_scenario,
      ~ run_paired_wilcox_time(.x, .y, df_tests)
    )
  ) %>%
  unnest(test_results) %>%
  mutate(
    p_value_holm     = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  select(sf, workers, strategy, distributed_scenario,
         n_pairs, median_baseline, median_distributed,
         p_value, p_value_holm, significant_holm,
         rank_biserial, effect_magnitude)

rq1_time_results

sf,workers,strategy,distributed_scenario,n_pairs,median_baseline,median_distributed,p_value,p_value_holm,significant_holm,rank_biserial,effect_magnitude
<dbl>,<dbl>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<effctsz_>
10,2,D1,SF10_N2_D1,225,9.564,3.238,6.865874e-10,1.373175e-09,TRUE,0.474,very large
10,2,D2,SF10_N2_D2,221,9.182,3.453,2.143948e-11,6.431844e-11,TRUE,0.520,very large
10,5,D1,SF10_N5_D1,225,9.564,3.662,4.176386e-07,4.176386e-07,TRUE,0.389,large
10,5,D2,SF10_N5_D2,221,9.182,3.159,1.450036e-16,5.800142e-16,TRUE,0.641,very large
50,2,D1,SF50_N2_D1,220,67.308,18.195,3.042819e-18,1.521409e-17,TRUE,0.677,very large
50,2,D2,SF50_N2_D2,217,58.728,16.005,3.564610e-24,3.208149e-23,TRUE,0.794,very large
50,5,D1,SF50_N5_D1,220,67.308,16.584,2.696834e-20,1.618100e-19,TRUE,0.718,very large
50,5,D2,SF50_N5_D2,217,58.728,14.641,2.063490e-28,2.476188e-27,TRUE,0.866,very large
100,2,D1,SF100_N2_D1,214,354.290,69.228,5.921984e-25,5.921984e-24,TRUE,0.813,very large


In [16]:
write_csv(
  rq1_time_results,
  file.path(tables_dir, "rq1_time_wilcoxon_paired.csv")
)

Testele Wilcoxon signed-rank aplicate pe perechi au respins H01 în toate cele 12 comparații (p-value post-corectie Holm mult mai mic de 0.05), cu mărimi de efect mari sau foarte mari (rank-biserial 0,389–0,866). Reducerea medianei timpului de execuție produsă de distribuire variază de la circa 60% la SF10 până la circa 80% la SF100, confirmând H11.

## B. Executabilitate: 12 teste McNemar pe `success`

Pentru axa de executabilitate, se compară variabila binară `success` între scenariul baseline și fiecare scenariu distribuit corespunzător aceluiași scale factor. 

Testul McNemar analizează perechile discordante: cazurile în care interogarea rulează cu succes în baseline, dar eșuează în scenariul distribuit, și cazurile inverse. Astfel, testul verifică dacă trecerea la arhitectura distribuită modifică semnificativ probabilitatea de succes a execuției.

In [19]:
# Funcție helper: rulează McNemar pentru success baseline vs distributed
run_paired_mcnemar_success <- function(baseline_scen, distrib_scen, df) {
  
  paired_df <- df %>%
    filter(scenario %in% c(baseline_scen, distrib_scen)) %>%
    select(query_id, scenario, success) %>%
    pivot_wider(names_from = scenario, values_from = success) %>%
    drop_na()
  
  baseline_success <- paired_df[[baseline_scen]]
  distrib_success  <- paired_df[[distrib_scen]]
  
  tab <- table(
    baseline = factor(baseline_success, levels = c(FALSE, TRUE)),
    distributed = factor(distrib_success, levels = c(FALSE, TRUE))
  )
  
  # Celule tabel 2x2
  both_fail <- tab["FALSE", "FALSE"]
  baseline_fail_distrib_success <- tab["FALSE", "TRUE"]
  baseline_success_distrib_fail <- tab["TRUE", "FALSE"]
  both_success <- tab["TRUE", "TRUE"]
  
  # McNemar se bazează pe perechile discordante
  test <- stats::mcnemar.test(tab, correct = FALSE)
  
  # Odds ratio pe perechile discordante:
  # > 1 => mai multe cazuri pierdute în distributed față de baseline
  # < 1 => mai multe cazuri recuperate în distributed
  discordant_or <- if_else(
    baseline_fail_distrib_success == 0,
    NA_real_,
    as.numeric(baseline_success_distrib_fail / baseline_fail_distrib_success)
  )
  
  tibble(
    n_pairs = nrow(paired_df),
    both_success = as.numeric(both_success),
    baseline_success_distributed_fail = as.numeric(baseline_success_distrib_fail),
    baseline_fail_distributed_success = as.numeric(baseline_fail_distrib_success),
    both_fail = as.numeric(both_fail),
    statistic_chi_square = unname(test$statistic),
    p_value = test$p.value,
    discordant_or = discordant_or
  )
}

In [20]:
rq1_status_results <- rq1_comparisons %>%
  mutate(
    test_results = map2(
      baseline_scenario, distributed_scenario,
      ~ run_paired_mcnemar_success(.x, .y, df_tests)
    )
  ) %>%
  unnest(test_results) %>%
  mutate(
    p_value_holm = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  select(
    sf, workers, strategy, distributed_scenario,
    n_pairs,
    both_success,
    baseline_success_distributed_fail,
    baseline_fail_distributed_success,
    both_fail,
    statistic_chi_square,
    p_value, p_value_holm, significant_holm,
    discordant_or
  )

In [21]:
write_csv(
  rq1_status_results,
  file.path(tables_dir, "rq1_status_mcnemar_baseline_vs_distributed.csv")
)

rq1_status_results

sf,workers,strategy,distributed_scenario,n_pairs,both_success,baseline_success_distributed_fail,baseline_fail_distributed_success,both_fail,statistic_chi_square,p_value,p_value_holm,significant_holm,discordant_or
<dbl>,<dbl>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>
10,2,D1,SF10_N2_D1,250,225,25,0,0,25.00000,5.733031e-07,2.293213e-06,TRUE,NA
10,2,D2,SF10_N2_D2,250,221,29,0,0,29.00000,7.237830e-08,7.237830e-07,TRUE,NA
10,5,D1,SF10_N5_D1,250,225,25,0,0,25.00000,5.733031e-07,2.293213e-06,TRUE,NA
10,5,D2,SF10_N5_D2,250,221,29,0,0,29.00000,7.237830e-08,7.237830e-07,TRUE,NA
50,2,D1,SF50_N2_D1,250,220,27,0,3,27.00000,2.034555e-07,1.332227e-06,TRUE,NA
50,2,D2,SF50_N2_D2,250,217,30,0,3,30.00000,4.320463e-08,5.184556e-07,TRUE,NA
50,5,D1,SF50_N5_D1,250,220,27,0,3,27.00000,2.034555e-07,1.332227e-06,TRUE,NA
50,5,D2,SF50_N5_D2,250,217,30,0,3,30.00000,4.320463e-08,5.184556e-07,TRUE,NA
100,2,D1,SF100_N2_D1,250,214,30,3,3,22.09091,2.600383e-06,5.200766e-06,TRUE,10.000000


Testele McNemar pentru RQ1 privind executabilitatea indică p-value-uri ajustate Holm sub 0,05 pentru toate cele 12 comparații baseline vs scenarii distribuite. Prin urmare, ipoteza nulă este respinsă și se poate concluziona că trecerea de la PostgreSQL mononod la scenariile distribuite Citus modifică semnificativ executabilitatea interogărilor.

Direcția diferenței este clară: în toate comparațiile există interogări care rulează cu succes în baseline, dar eșuează în scenariile distribuite (`baseline_success_distributed_fail` între 25 și 31), în timp ce situațiile inverse sunt inexistente sau foarte rare (`baseline_fail_distributed_success` între 0 și 3). Astfel, deși scenariile distribuite au redus semnificativ timpii pentru interogările reușite, ele au introdus și o scădere semnificativă a ratei de succes față de baseline.

**Concluzie pentru RQ1**

Rezultatele inferențiale pentru RQ1 arată că trecerea de la PostgreSQL mononod la arhitectura distribuită Citus are un impact semnificativ atât asupra timpului de execuție, cât și asupra executabilității interogărilor.

Pe axa timpului de execuție, testele Wilcoxon signed-rank au indicat diferențe semnificative statistic pentru toate cele 12 comparații baseline vs scenarii distribuite, după aplicarea corecției Holm. Prin urmare, H01 este respinsă pentru dimensiunea temporală, iar rezultatele susțin H11. Direcția diferenței, interpretată prin medianele timpilor, indică faptul că scenariile distribuite reduc timpul de execuție pentru interogările care rulează cu succes în ambele configurații.

Pe axa executabilității, testele McNemar au indicat, de asemenea, diferențe semnificative statistic între baseline și scenariile distribuite. Totuși, direcția efectului este diferită: scenariile distribuite pierd un număr de interogări care rulau cu succes în baseline, în timp ce cazurile inverse sunt inexistente sau foarte rare. Astfel, **arhitectura distribuită aduce un câștig de performanță pentru interogările executabile, dar cu un cost la nivelul ratei de succes**.

# RQ2 - Scalare de la N2 la N5

RQ2. Care este impactul adăugării de noi noduri de procesare, prin scalarea orizontală de la 2 la 5 workeri, asupra performanței clusterului în gestionarea sarcinilor de lucru de tip TPC-DS?

H02: Creșterea numărului de workeri de la 2 la 5 nu produce diferențe semnificative asupra timpilor de execuție ai interogărilor analitice.

H12: Extinderea clusterului de la 2 la 5 workeri reduce semnificativ timpul de execuție al interogărilor analitice. 

  - Timp: 6 teste Wilcoxon signed-rank pereche (2 strategii × 3 SF), **unilateral** (H12 direcțional: N5 reduce timpul)
  - Executabilitate: 6 teste McNemar pe `success` (2 strategii × 3 SF)


## A. Timpi de execuție - 6 teste Wilcoxon signed-rank pereche, unilateral

In [22]:
# Funcție helper: Wilcoxon signed-rank pereche pentru N2 vs N5
run_paired_wilcox_n2_n5 <- function(n2_scen, n5_scen, df) {
  
  paired_df <- df %>%
    filter(scenario %in% c(n2_scen, n5_scen)) %>%
    filter(status == "SUCCESS") %>%
    select(query_id, scenario, elapsed_sec) %>%
    pivot_wider(names_from = scenario, values_from = elapsed_sec) %>%
    drop_na()
  
  n2_times <- paired_df[[n2_scen]]
  n5_times <- paired_df[[n5_scen]]
  
  test <- wilcox.test(
    n2_times, n5_times,
    paired = TRUE,
    alternative = "greater",  # H12: N2 > N5, deci N5 reduce timpul
    conf.int = TRUE,
    exact = FALSE
  )
  
  eff <- effectsize::rank_biserial(n2_times, n5_times, paired = TRUE)
  
  tibble(
    n_pairs = nrow(paired_df),
    median_n2 = round(median(n2_times), 3),
    median_n5 = round(median(n5_times), 3),
    node_scaling_speedup = round(median(n2_times) / median(n5_times), 3),
    time_reduction_pct = round((median(n2_times) - median(n5_times)) / median(n2_times) * 100, 2),
    scaling_efficiency = round((median(n2_times) / median(n5_times)) / 2.5, 3),
    statistic_V = unname(test$statistic),
    p_value = test$p.value,
    rank_biserial = round(eff$r_rank_biserial, 3),
    effect_magnitude = effectsize::interpret_rank_biserial(eff$r_rank_biserial)
  )
}

In [23]:
# Cele 6 comparații RQ2: N2 vs N5 pentru fiecare SF și strategie
rq2_comparisons <- expand_grid(
  sf = c(10, 50, 100),
  strategy = c("D1", "D2")
) %>%
  mutate(
    n2_scenario = paste0("SF", sf, "_N2_", strategy),
    n5_scenario = paste0("SF", sf, "_N5_", strategy)
  )

rq2_comparisons

sf,strategy,n2_scenario,n5_scenario
<dbl>,<chr>,<chr>,<chr>
10,D1,SF10_N2_D1,SF10_N5_D1
10,D2,SF10_N2_D2,SF10_N5_D2
50,D1,SF50_N2_D1,SF50_N5_D1
50,D2,SF50_N2_D2,SF50_N5_D2
100,D1,SF100_N2_D1,SF100_N5_D1
100,D2,SF100_N2_D2,SF100_N5_D2


In [24]:
rq2_time_results <- rq2_comparisons %>%
  mutate(
    test_results = map2(
      n2_scenario, n5_scenario,
      ~ run_paired_wilcox_n2_n5(.x, .y, df_tests)
    )
  ) %>%
  unnest(test_results) %>%
  mutate(
    p_value_holm = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  select(
    sf, strategy, n2_scenario, n5_scenario,
    n_pairs, median_n2, median_n5,
    node_scaling_speedup, time_reduction_pct, scaling_efficiency,
    p_value, p_value_holm, significant_holm,
    rank_biserial, effect_magnitude
  )

write_csv(
  rq2_time_results,
  file.path(tables_dir, "rq2_time_wilcoxon_n2_vs_n5.csv")
)

rq2_time_results

sf,strategy,n2_scenario,n5_scenario,n_pairs,median_n2,median_n5,node_scaling_speedup,time_reduction_pct,scaling_efficiency,p_value,p_value_holm,significant_holm,rank_biserial,effect_magnitude
<dbl>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<effctsz_>
10,D1,SF10_N2_D1,SF10_N5_D1,225,3.238,3.662,0.884,-13.09,0.354,1.000000e+00,1.000000e+00,FALSE,-0.469,very large
10,D2,SF10_N2_D2,SF10_N5_D2,221,3.453,3.159,1.093,8.51,0.437,3.830987e-02,1.532395e-01,FALSE,0.137,small
50,D1,SF50_N2_D1,SF50_N5_D1,220,18.195,16.584,1.097,8.86,0.439,1.749963e-03,8.749814e-03,TRUE,0.227,medium
50,D2,SF50_N2_D2,SF50_N5_D2,217,16.005,14.641,1.093,8.52,0.437,5.400563e-18,3.240338e-17,TRUE,0.671,very large
100,D1,SF100_N2_D1,SF100_N5_D1,217,71.553,72.493,0.987,-1.31,0.395,9.986068e-01,1.000000e+00,FALSE,-0.234,medium
100,D2,SF100_N2_D2,SF100_N5_D2,214,62.991,82.236,0.766,-30.55,0.306,9.999785e-01,1.000000e+00,FALSE,-0.322,large


Rezultatele testelor Wilcoxon signed-rank pentru RQ2 indică un efect neuniform al scalării de la 2 la 5 workeri. După aplicarea corecției Holm, doar comparațiile de la SF50 sunt semnificative statistic: SF50_D1 (`p_holm = 0,0087`) și SF50_D2 (`p_holm < 0,001`). În aceste două cazuri, N5 reduce timpul median față de N2, cu speedup median de aproximativ 1,097 pentru D1 și 1,093 pentru D2.

Pentru SF10 și SF100, diferențele nu sunt semnificative statistic după corecția Holm. Mai mult, în unele cazuri, mediana timpului este mai mare pentru N5 decât pentru N2, de exemplu SF10_D1 și SF100_D2, unde `node_scaling_speedup` este sub 1. Acest lucru arată că adăugarea de workeri nu produce automat o îmbunătățire a timpului de execuție.

Prin urmare, **H12 este susținută doar parțial**, în special pentru SF50, unde scalarea de la 2 la 5 workeri reduce semnificativ timpul de execuție. La nivel general, H02 nu poate fi respinsă pentru toate configurațiile, deoarece efectul scalării este dependent de scale factor și de strategia de distribuire.

## B. Executabilitate: 6 teste McNemar pe `success`

Pentru executabilitate, se compară variabila binară `success` între configurațiile N2 și N5, pentru același scale factor, aceeași strategie și același `query_id`.

Deoarece aceeași interogare este rulată în ambele configurații, se utilizează testul McNemar. Testul verifică dacă trecerea de la 2 la 5 workeri modifică semnificativ probabilitatea de succes a execuției.

In [26]:
# Funcție helper: McNemar pentru success N2 vs N5
run_paired_mcnemar_n2_n5 <- function(n2_scen, n5_scen, df) {
  
  paired_df <- df %>%
    filter(scenario %in% c(n2_scen, n5_scen)) %>%
    select(query_id, scenario, success) %>%
    pivot_wider(names_from = scenario, values_from = success) %>%
    drop_na()
  
  n2_success <- paired_df[[n2_scen]]
  n5_success <- paired_df[[n5_scen]]
  
  tab <- table(
    N2 = factor(n2_success, levels = c(FALSE, TRUE)),
    N5 = factor(n5_success, levels = c(FALSE, TRUE))
  )
  
  both_fail <- tab["FALSE", "FALSE"]
  n2_fail_n5_success <- tab["FALSE", "TRUE"]
  n2_success_n5_fail <- tab["TRUE", "FALSE"]
  both_success <- tab["TRUE", "TRUE"]
  
  test <- stats::mcnemar.test(tab, correct = FALSE)
  
  discordant_or <- if_else(
    n2_fail_n5_success == 0,
    NA_real_,
    as.numeric(n2_success_n5_fail / n2_fail_n5_success)
  )
  
  tibble(
    n_pairs = nrow(paired_df),
    both_success = as.numeric(both_success),
    n2_success_n5_fail = as.numeric(n2_success_n5_fail),
    n2_fail_n5_success = as.numeric(n2_fail_n5_success),
    both_fail = as.numeric(both_fail),
    statistic_chi_square = unname(test$statistic),
    p_value = test$p.value,
    discordant_or = discordant_or
  )
}

In [27]:
rq2_status_results <- rq2_comparisons %>%
  mutate(
    test_results = map2(
      n2_scenario, n5_scenario,
      ~ run_paired_mcnemar_n2_n5(.x, .y, df_tests)
    )
  ) %>%
  unnest(test_results) %>%
  mutate(
    p_value_holm = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  select(
    sf, strategy, n2_scenario, n5_scenario,
    n_pairs,
    both_success,
    n2_success_n5_fail,
    n2_fail_n5_success,
    both_fail,
    statistic_chi_square,
    p_value, p_value_holm, significant_holm,
    discordant_or
  )

write_csv(
  rq2_status_results,
  file.path(tables_dir, "rq2_status_mcnemar_n2_vs_n5.csv")
)

rq2_status_results

sf,strategy,n2_scenario,n5_scenario,n_pairs,both_success,n2_success_n5_fail,n2_fail_n5_success,both_fail,statistic_chi_square,p_value,p_value_holm,significant_holm,discordant_or
<dbl>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>
10,D1,SF10_N2_D1,SF10_N5_D1,250,225,0,0,25,NaN,NaN,NaN,NA,NA
10,D2,SF10_N2_D2,SF10_N5_D2,250,221,0,0,29,NaN,NaN,NaN,NA,NA
50,D1,SF50_N2_D1,SF50_N5_D1,250,220,0,0,30,NaN,NaN,NaN,NA,NA
50,D2,SF50_N2_D2,SF50_N5_D2,250,217,0,0,33,NaN,NaN,NaN,NA,NA
100,D1,SF100_N2_D1,SF100_N5_D1,250,217,0,1,32,1,0.3173105,0.634621,FALSE,0
100,D2,SF100_N2_D2,SF100_N5_D2,250,214,1,0,35,1,0.3173105,0.634621,FALSE,NA


Rezultatele testelor McNemar pentru RQ2 indică faptul că scalarea de la 2 la 5 workeri nu produce o schimbare semnificativă a executabilității interogărilor. În patru dintre cele șase comparații nu există perechi discordante, ceea ce arată că N2 și N5 au exact același comportament din perspectiva succesului/eșecului. În celelalte două comparații, diferențele sunt punctuale și nesemnificative statistic după corecția Holm.

Prin urmare, **pe axa executabilității, H02 nu poate fi respinsă**. Adăugarea de workeri nu afectează semnificativ rata de succes a interogărilor în scenariile analizate.

# RQ3 - Strategie D1 vs D2

RQ3. Cum influențează strategia de distribuire a datelor, respectiv alegerea cheii de distribuire, timpul de execuție și rata de succes a interogărilor?

H03: Alegerea strategiei de distribuire a datelor, D1 sau D2, nu influențează semnificativ timpii de execuție sau rata de succes a interogărilor.

H13: Alegerea strategiei de distribuire a datelor influențează semnificativ timpii de execuție sau rata de succes a interogărilor, în funcție de modul în care cheia de distribuire favorizează colocarea datelor.


- Timp: 6 teste Wilcoxon signed-rank pereche (2 niveluri workers × 3 SF), **bilateral** (H13 nondirecțional)
- Executabilitate: 6 teste McNemar pe `success` (2 × 3 SF)

## A. Timpi de execuție - 6 teste Wilcoxon signed-rank pereche

Pentru RQ3, timpul de execuție este comparat între strategiile D1 și D2, pentru același scale factor, același număr de workeri și același `query_id`.

Sunt incluse doar interogările executate cu succes în ambele strategii comparate. Deoarece H13 nu presupune din start că una dintre strategii este întotdeauna mai rapidă, se utilizează testul Wilcoxon signed-rank pereche, bilateral.

In [28]:
# Funcție helper: Wilcoxon signed-rank pereche pentru D1 vs D2
run_paired_wilcox_d1_d2 <- function(d1_scen, d2_scen, df) {
  
  paired_df <- df %>%
    filter(scenario %in% c(d1_scen, d2_scen)) %>%
    filter(status == "SUCCESS") %>%
    select(query_id, scenario, elapsed_sec) %>%
    pivot_wider(names_from = scenario, values_from = elapsed_sec) %>%
    drop_na()
  
  d1_times <- paired_df[[d1_scen]]
  d2_times <- paired_df[[d2_scen]]
  
  test <- wilcox.test(
    d1_times, d2_times,
    paired = TRUE,
    alternative = "two.sided",
    conf.int = TRUE,
    exact = FALSE
  )
  
  eff <- effectsize::rank_biserial(d1_times, d2_times, paired = TRUE)
  
  tibble(
    n_pairs = nrow(paired_df),
    median_d1 = round(median(d1_times), 3),
    median_d2 = round(median(d2_times), 3),
    median_ratio_d2_over_d1 = round(median(d2_times) / median(d1_times), 3),
    statistic_V = unname(test$statistic),
    p_value = test$p.value,
    rank_biserial = round(eff$r_rank_biserial, 3),
    effect_magnitude = effectsize::interpret_rank_biserial(eff$r_rank_biserial)
  )
}

In [29]:
# Cele 6 comparații RQ3: D1 vs D2 pentru fiecare SF și număr de workeri
rq3_comparisons <- expand_grid(
  sf = c(10, 50, 100),
  workers = c(2, 5)
) %>%
  mutate(
    d1_scenario = paste0("SF", sf, "_N", workers, "_D1"),
    d2_scenario = paste0("SF", sf, "_N", workers, "_D2")
  )

rq3_comparisons

sf,workers,d1_scenario,d2_scenario
<dbl>,<dbl>,<chr>,<chr>
10,2,SF10_N2_D1,SF10_N2_D2
10,5,SF10_N5_D1,SF10_N5_D2
50,2,SF50_N2_D1,SF50_N2_D2
50,5,SF50_N5_D1,SF50_N5_D2
100,2,SF100_N2_D1,SF100_N2_D2
100,5,SF100_N5_D1,SF100_N5_D2


In [30]:
rq3_time_results <- rq3_comparisons %>%
  mutate(
    test_results = map2(
      d1_scenario, d2_scenario,
      ~ run_paired_wilcox_d1_d2(.x, .y, df_tests)
    )
  ) %>%
  unnest(test_results) %>%
  mutate(
    p_value_holm = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  select(
    sf, workers, d1_scenario, d2_scenario,
    n_pairs, median_d1, median_d2, median_ratio_d2_over_d1,
    p_value, p_value_holm, significant_holm,
    rank_biserial, effect_magnitude
  )

write_csv(
  rq3_time_results,
  file.path(tables_dir, "rq3_time_wilcoxon_d1_vs_d2.csv")
)

rq3_time_results

sf,workers,d1_scenario,d2_scenario,n_pairs,median_d1,median_d2,median_ratio_d2_over_d1,p_value,p_value_holm,significant_holm,rank_biserial,effect_magnitude
<dbl>,<dbl>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<effctsz_>
10,2,SF10_N2_D1,SF10_N2_D2,216,3.239,3.511,1.084,1.323200e-01,2.646399e-01,FALSE,0.118,small
10,5,SF10_N5_D1,SF10_N5_D2,216,3.663,3.180,0.868,1.026087e-11,5.130433e-11,TRUE,0.535,very large
50,2,SF50_N2_D1,SF50_N2_D2,211,18.048,16.005,0.887,1.124545e-05,4.498181e-05,TRUE,0.349,large
50,5,SF50_N5_D1,SF50_N5_D2,211,16.526,13.546,0.820,3.026967e-15,1.816180e-14,TRUE,0.626,very large
100,2,SF100_N2_D1,SF100_N2_D2,207,68.239,62.865,0.921,7.252742e-02,2.175823e-01,FALSE,0.144,small
100,5,SF100_N5_D1,SF100_N5_D2,208,73.499,82.923,1.128,4.269017e-01,4.269017e-01,FALSE,-0.064,very small


Rezultatele testelor Wilcoxon signed-rank pentru RQ3 indică faptul că strategia de distribuire influențează semnificativ timpul de execuție doar în anumite configurații. După aplicarea corecției Holm, diferențe semnificative apar pentru SF10_N5, SF50_N2 și SF50_N5. În toate aceste cazuri, `median_ratio_d2_over_d1` este sub 1, ceea ce indică faptul că D2 are timp median mai mic decât D1.

Pentru SF10_N2, SF100_N2 și SF100_N5, diferențele nu sunt semnificative statistic după corecția Holm. La SF100_N5, raportul median este peste 1 (`1,128`), ceea ce sugerează chiar o tendință inversă, dar nesemnificativă statistic.

Prin urmare, **H13 este susținută parțial pe axa timpului de execuție**. Strategia D2 produce îmbunătățiri semnificative în unele configurații, mai ales la SF50 și la SF10 cu 5 workeri, însă efectul nu este uniform pentru toate volumele de date și configurațiile de cluster.

## B. Executabilitate - 6 teste McNemar pe `success`

Pentru executabilitate, se compară variabila binară `success` între strategiile D1 și D2, pentru același scale factor, același număr de workeri și același `query_id`.

Testul McNemar verifică dacă alegerea strategiei de distribuire modifică semnificativ probabilitatea de succes a execuției.

In [31]:
# Funcție helper: McNemar pentru success D1 vs D2
run_paired_mcnemar_d1_d2 <- function(d1_scen, d2_scen, df) {
  
  paired_df <- df %>%
    filter(scenario %in% c(d1_scen, d2_scen)) %>%
    select(query_id, scenario, success) %>%
    pivot_wider(names_from = scenario, values_from = success) %>%
    drop_na()
  
  d1_success <- paired_df[[d1_scen]]
  d2_success <- paired_df[[d2_scen]]
  
  tab <- table(
    D1 = factor(d1_success, levels = c(FALSE, TRUE)),
    D2 = factor(d2_success, levels = c(FALSE, TRUE))
  )
  
  both_fail <- tab["FALSE", "FALSE"]
  d1_fail_d2_success <- tab["FALSE", "TRUE"]
  d1_success_d2_fail <- tab["TRUE", "FALSE"]
  both_success <- tab["TRUE", "TRUE"]
  
  if ((d1_fail_d2_success + d1_success_d2_fail) == 0) {
    statistic <- NA_real_
    p_val <- NA_real_
  } else {
    test <- stats::mcnemar.test(tab, correct = FALSE)
    statistic <- unname(test$statistic)
    p_val <- test$p.value
  }
  
  discordant_or <- if_else(
    d1_fail_d2_success == 0,
    NA_real_,
    as.numeric(d1_success_d2_fail / d1_fail_d2_success)
  )
  
  tibble(
    n_pairs = nrow(paired_df),
    both_success = as.numeric(both_success),
    d1_success_d2_fail = as.numeric(d1_success_d2_fail),
    d1_fail_d2_success = as.numeric(d1_fail_d2_success),
    both_fail = as.numeric(both_fail),
    statistic_chi_square = statistic,
    p_value = p_val,
    discordant_or = discordant_or
  )
}

In [32]:
rq3_status_results <- rq3_comparisons %>%
  mutate(
    test_results = map2(
      d1_scenario, d2_scenario,
      ~ run_paired_mcnemar_d1_d2(.x, .y, df_tests)
    )
  ) %>%
  unnest(test_results) %>%
  mutate(
    p_value_holm = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  select(
    sf, workers, d1_scenario, d2_scenario,
    n_pairs,
    both_success,
    d1_success_d2_fail,
    d1_fail_d2_success,
    both_fail,
    statistic_chi_square,
    p_value, p_value_holm, significant_holm,
    discordant_or
  )

write_csv(
  rq3_status_results,
  file.path(tables_dir, "rq3_status_mcnemar_d1_vs_d2.csv")
)

rq3_status_results

sf,workers,d1_scenario,d2_scenario,n_pairs,both_success,d1_success_d2_fail,d1_fail_d2_success,both_fail,statistic_chi_square,p_value,p_value_holm,significant_holm,discordant_or
<dbl>,<dbl>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>
10,2,SF10_N2_D1,SF10_N2_D2,250,216,9,5,20,1.1428571,0.2850494,1,FALSE,1.800000
10,5,SF10_N5_D1,SF10_N5_D2,250,216,9,5,20,1.1428571,0.2850494,1,FALSE,1.800000
50,2,SF50_N2_D1,SF50_N2_D2,250,211,9,6,24,0.6000000,0.4385780,1,FALSE,1.500000
50,5,SF50_N5_D1,SF50_N5_D2,250,211,9,6,24,0.6000000,0.4385780,1,FALSE,1.500000
100,2,SF100_N2_D1,SF100_N2_D2,250,207,10,8,25,0.2222222,0.6373519,1,FALSE,1.250000
100,5,SF100_N5_D1,SF100_N5_D2,250,208,10,6,26,1.0000000,0.3173105,1,FALSE,1.666667


Rezultatele testelor McNemar pentru RQ3 indică faptul că alegerea strategiei de distribuire nu modifică semnificativ executabilitatea interogărilor. În toate cele 6 comparații D1 vs D2, p-value-ul ajustat Holm este egal cu 1, iar `significant_holm = FALSE`.

Deși există perechi discordante în ambele direcții, diferențele sunt relativ echilibrate. De exemplu, numărul de interogări care rulează în D1 dar eșuează în D2 variază între 9 și 10, iar numărul celor care eșuează în D1 dar rulează în D2 variază între 5 și 8. Acest dezechilibru nu este suficient pentru a susține o diferență semnificativă statistic.

Prin urmare, pentru axa executabilității, **H03 nu poate fi respinsă**. Strategia de distribuire influențează parțial timpul de execuție, dar nu produce o modificare semnificativă a ratei de succes în scenariile analizate.

# RQ4 - Caracteristici structurale ale interogarilor SQL 

RQ4. Cum influențează caracteristicile structurale ale interogărilor SQL performanța și executabilitatea acestora în scenarii distribuite?

H04: Caracteristicile structurale ale interogărilor SQL nu influențează semnificativ performanța sau executabilitatea acestora în scenariile distribuite.

H14: Anumite caracteristici structurale ale interogărilor SQL sunt asociate semnificativ cu variații ale timpilor de execuție sau ale executabilității în scenarii distribuite 


- Timp: teste Mann-Whitney U / Wilcoxon rank-sum pentru relația dintre prezența unei caracteristici SQL (`yes` / `no`) și `log_elapsed_sec`, aplicate doar pe scenariile distribuite cu `SUCCESS`.

- Timp: corelații Spearman între caracteristicile numerice ale interogărilor (`n_tables`, `n_joins`, `n_where_predicates`, `n_aggregate_calls` etc.) și `log_elapsed_sec`, aplicate doar pe scenariile distribuite cu `SUCCESS`.

- Executabilitate: teste Fisher exact pentru asocierea dintre prezența unei caracteristici SQL (`yes` / `no`) și variabila `success`, aplicate pe toate execuțiile distribuite.

## A. Timpi de execuție - variabile binare `has_*` - Teste Wilcoxon rank-sum

Pentru prima parte a RQ4 se testează dacă prezența unor caracteristici structurale SQL este asociată cu diferențe semnificative ale timpului de execuție. Analiza este aplicată doar pe scenariile distribuite și doar pe interogările executate cu succes (`SUCCESS`), deoarece timpul de execuție este interpretabil doar pentru execuțiile finalizate.

Pentru fiecare variabilă binară `has_*`, sunt comparate două grupuri independente de execuții: interogări în care caracteristica este prezentă (`yes`) și interogări în care caracteristica este absentă (`no`).

In [35]:
df_rq4_time <- df_tests %>%
  filter(
    architecture == "distributed",
    success == TRUE
  )

has_vars <- names(df_rq4_time)[startsWith(names(df_rq4_time), "has_")]

# Selectăm doar variabilele has_* cu minimum 10 observații în ambele grupuri
has_vars_useful <- df_rq4_time %>%
  select(all_of(has_vars)) %>%
  pivot_longer(
    cols = everything(),
    names_to = "characteristic",
    values_to = "presence"
  ) %>%
  count(characteristic, presence) %>%
  pivot_wider(
    names_from = presence,
    values_from = n,
    values_fill = 0
  ) %>%
  filter(no >= 10, yes >= 10) %>%
  pull(characteristic)

has_vars_useful

[1] "has_aggregates" "has_case"       "has_cross_join" "has_cte"       
 [5] "has_distinct"   "has_except"     "has_full_join"  "has_group_by"  
 [9] "has_having"     "has_intersect"  "has_left_join"  "has_limit"     
[13] "has_order_by"   "has_right_join" "has_subqueries" "has_union"     
[17] "has_window"

Au fost păstrate doar variabilele `has_*` care au cel puțin 10 observații în ambele categorii (`no` și `yes`). Această filtrare evită testele instabile pentru caracteristici foarte rare, precum `has_cube`, `has_rollup` sau `has_grouping`.

In [38]:
# Funcție helper: Mann-Whitney U / Wilcoxon rank-sum pentru has_* vs log_elapsed_sec

run_mann_whitney_has_time <- function(var_name, df) {
  
  test_df <- df %>%
    select(log_elapsed_sec, all_of(var_name)) %>%
    rename(presence = all_of(var_name)) %>%
    drop_na() %>%
    mutate(presence = factor(presence, levels = c("no", "yes")))
  
  test <- wilcox.test(
    log_elapsed_sec ~ presence,
    data = test_df,
    exact = FALSE,
    conf.int = TRUE
  )
  
  eff <- effectsize::rank_biserial(
    log_elapsed_sec ~ presence,
    data = test_df
  )
  
  summary_df <- test_df %>%
    group_by(presence) %>%
    summarise(
      n = n(),
      median_log_elapsed_sec = median(log_elapsed_sec, na.rm = TRUE),
      median_elapsed_sec = median(10^log_elapsed_sec, na.rm = TRUE),
      .groups = "drop"
    )
  
  median_no <- summary_df$median_elapsed_sec[summary_df$presence == "no"]
  median_yes <- summary_df$median_elapsed_sec[summary_df$presence == "yes"]
  
  tibble(
    characteristic = var_name,
    n_no = summary_df$n[summary_df$presence == "no"],
    n_yes = summary_df$n[summary_df$presence == "yes"],
    median_log_no = round(summary_df$median_log_elapsed_sec[summary_df$presence == "no"], 3),
    median_log_yes = round(summary_df$median_log_elapsed_sec[summary_df$presence == "yes"], 3),
    median_sec_no = round(median_no, 3),
    median_sec_yes = round(median_yes, 3),
    median_ratio_yes_over_no = round(median_yes / median_no, 3),
    statistic_w = unname(test$statistic),
    p_value = test$p.value,
    rank_biserial = round(eff$r_rank_biserial, 3),
    effect_magnitude = effectsize::interpret_rank_biserial(eff$r_rank_biserial)
  )
}

In [40]:
rq4_time_has_results <- map_dfr(
  has_vars_useful,
  ~ run_mann_whitney_has_time(.x, df_rq4_time)
) %>%
  mutate(
    p_value_holm = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  arrange(p_value_holm)

write_csv(
  rq4_time_has_results,
  file.path(tables_dir, "rq4_time_mann_whitney_has_characteristics.csv")
)

rq4_time_has_results

characteristic,n_no,n_yes,median_log_no,median_log_yes,median_sec_no,median_sec_yes,median_ratio_yes_over_no,statistic_w,p_value,rank_biserial,effect_magnitude,p_value_holm,significant_holm
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<effctsz_>,<dbl>,<lgl>
has_union,2518,112,1.179,1.920,15.110,83.280,5.512,74807.5,3.802891e-17,-0.469,very large,6.464915e-16,TRUE
has_left_join,2137,493,1.165,1.384,14.631,24.197,1.654,422297.5,6.244739e-12,-0.198,small,9.991582e-11,TRUE
has_group_by,238,2392,1.485,1.177,30.541,15.018,0.492,353360.5,7.735428e-10,0.241,medium,1.160314e-08,TRUE
has_right_join,2614,16,1.198,2.908,15.791,809.770,51.282,2462.0,1.110977e-09,-0.882,very large,1.555367e-08,TRUE
has_aggregates,244,2386,1.479,1.176,30.163,15.011,0.498,358823.5,2.035136e-09,0.233,medium,2.645676e-08,TRUE
has_subqueries,2109,521,1.170,1.367,14.804,23.272,1.572,463351.0,2.963864e-08,-0.157,small,3.556637e-07,TRUE
has_order_by,397,2233,1.386,1.174,24.340,14.938,0.614,518674.5,6.302472e-08,0.170,small,6.932719e-07,TRUE
has_intersect,2546,84,1.189,1.916,15.446,82.435,5.337,70709.5,1.224947e-07,-0.339,large,1.224947e-06,TRUE
has_distinct,2012,618,1.168,1.320,14.719,20.891,1.419,539040.0,5.535384e-07,-0.133,small,4.981846e-06,TRUE


Rezultatele testelor Mann-Whitney/Wilcoxon rank-sum arată că mai multe caracteristici structurale binare sunt asociate semnificativ cu diferențe ale timpului de execuție în scenariile distribuite cu `SUCCESS`. După corecția Holm, 12 din cele 17 caracteristici testate rămân semnificative statistic, lucru care **susține parțial H14 pe axa timpului de execuție**.

Cele mai clare creșteri ale timpului apar pentru `has_union`, `has_right_join`, `has_intersect`, `has_except`, `has_left_join`, `has_subqueries` și `has_distinct`, unde `median_ratio_yes_over_no` este peste 1. De exemplu, interogările cu `UNION` au un timp median de 83,280 secunde față de 15,110 secunde pentru cele fără `UNION`, iar interogările cu `RIGHT JOIN` au un timp median de 809,770 secunde față de 15,791 secunde. Totuși, pentru `RIGHT JOIN`, numărul de cazuri este foarte mic (`n_yes = 16`), deci rezultatul trebuie interpretat cu prudență.

În schimb, caracteristici precum `has_group_by`, `has_aggregates`, `has_order_by` și `has_full_join` au `median_ratio_yes_over_no` sub 1, ceea ce indică timpi mediani mai mici pentru interogările în care aceste construcții sunt prezente. De exemplu, interogările cu `GROUP BY` au o mediană de 15,018 secunde, comparativ cu 30,541 secunde pentru cele fără `GROUP BY`.

Pentru `has_having`, `has_window`, `has_cross_join`, `has_case` și `has_limit`, diferențele nu rămân semnificative după corecția Holm. Prin urmare, pentru aceste caracteristici nu există suficiente dovezi statistice că prezența lor este asociată cu modificări ale timpului de execuție în scenariile distribuite analizate.

În ansamblu, rezultatele arată că anumite construcții SQL sunt asociate semnificativ cu variații ale timpilor de execuție, dar direcția efectului nu este uniformă.

## B. Timpi de execuție - variabile numerice `n_*` - corelația Spearman

Pentru a doua parte a RQ4 se testează dacă variabilele numerice care descriu complexitatea structurală a interogărilor sunt asociate cu timpul de execuție. Analiza este aplicată doar pe scenariile distribuite cu `SUCCESS`.

In [42]:
numeric_vars_rq4 <- c(
  "n_tables",
  "n_joins",
  "n_inner_joins",
  "n_left_joins",
  "n_right_joins",
  "n_full_joins",
  "n_cross_joins",
  "n_subqueries",
  "n_ctes",
  "n_aggregate_types",
  "n_aggregate_calls",
  "n_where_predicates",
  "n_having_predicates"
)

run_spearman_numeric_time <- function(var_name, df) {
  
  test_df <- df %>%
    select(log_elapsed_sec, all_of(var_name)) %>%
    rename(value = all_of(var_name)) %>%
    drop_na()
  
  test <- suppressWarnings(
    cor.test(
      test_df$value,
      test_df$log_elapsed_sec,
      method = "spearman",
      exact = FALSE
    )
  )
  
  tibble(
    characteristic = var_name,
    n = nrow(test_df),
    spearman_rho = round(unname(test$estimate), 3),
    p_value = test$p.value,
    effect_magnitude = effectsize::interpret_r(unname(test$estimate))
  )
}

In [43]:
rq4_time_numeric_results <- map_dfr(
  numeric_vars_rq4,
  ~ run_spearman_numeric_time(.x, df_rq4_time)
) %>%
  mutate(
    p_value_holm = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  arrange(p_value_holm)

write_csv(
  rq4_time_numeric_results,
  file.path(tables_dir, "rq4_time_spearman_numeric_characteristics.csv")
)

rq4_time_numeric_results

characteristic,n,spearman_rho,p_value,effect_magnitude,p_value_holm,significant_holm
<chr>,<int>,<dbl>,<dbl>,<effctsz_>,<dbl>,<lgl>
n_left_joins,2630,0.144,1.226210e-13,small,1.594072e-12,TRUE
n_joins,2630,0.130,1.843484e-11,small,2.212181e-10,TRUE
n_tables,2630,0.122,3.090818e-10,small,3.399900e-09,TRUE
n_right_joins,2630,0.119,9.796808e-10,small,9.796808e-09,TRUE
n_aggregate_types,2630,-0.106,4.817398e-08,small,4.335658e-07,TRUE
n_subqueries,2630,0.101,2.354061e-07,small,1.883249e-06,TRUE
n_inner_joins,2630,0.072,2.169340e-04,very small,1.518538e-03,TRUE
n_full_joins,2630,-0.067,5.891325e-04,very small,3.534795e-03,TRUE
n_ctes,2630,0.060,1.988564e-03,very small,9.942821e-03,TRUE


Rezultatele corelațiilor Spearman pentru RQ4 arată că mai multe caracteristici numerice sunt asociate semnificativ statistic cu timpul de execuție în scenariile distribuite cu `SUCCESS`. După corecția Holm, 9 din cele 13 variabile analizate rămân semnificative statistic.

Totuși, magnitudinea efectelor este redusă. Cele mai mari corelații pozitive apar pentru `n_left_joins` (`rho = 0,144`), `n_joins` (`rho = 0,130`), `n_tables` (`rho = 0,122`) și `n_right_joins` (`rho = 0,119`), toate având efecte mici. Acest lucru indică faptul că **interogările cu mai multe tabele și join-uri tind să aibă timpi de execuție ușor mai mari, dar relația este slabă**.

Singura asociere negativă semnificativă mai vizibilă este pentru `n_aggregate_types` (`rho = -0,106`), ceea ce sugerează că un număr mai mare de tipuri de agregări este asociat cu timpi ușor mai mici, însă și aici efectul este mic. Variabile precum `n_cross_joins`, `n_having_predicates`, `n_aggregate_calls` și `n_where_predicates` nu rămân semnificative după corecția Holm.

În ansamblu, **H14 este susținută statistic pentru o parte dintre variabilele numerice**, dar efectele sunt slabe. Prin urmare, **complexitatea numerică a interogării contribuie la variația timpilor de execuție, însă nu explică singură performanța**; aceasta trebuie interpretată împreună cu arhitectura, strategia de distribuire și forma concretă a query-ului.

## C. Executabilitate — variabile binare has_* - Fisher exact test

Se testează dacă prezența unei caracteristici structurale SQL este asociată cu executabilitatea interogării în scenariile distribuite.

Analiza este aplicată pe toate execuțiile distribuite, indiferent de status. Variabila dependentă este `success`, iar pentru fiecare caracteristică `has_*` se compară distribuția succes/eșec între categoriile `yes` și `no`.

Se utilizează Fisher exact test, deoarece unele caracteristici structurale apar rar și pot genera frecvențe mici sau zero.

In [45]:
df_rq4_exec <- df_tests %>%
  filter(architecture == "distributed")

has_vars_exec <- names(df_rq4_exec)[startsWith(names(df_rq4_exec), "has_")]

run_fisher_has_success <- function(var_name, df) {
  
  test_df <- df %>%
    select(success, all_of(var_name)) %>%
    rename(presence = all_of(var_name)) %>%
    drop_na() %>%
    mutate(
      presence = factor(presence, levels = c("no", "yes")),
      success = factor(success, levels = c(FALSE, TRUE))
    )
  
  tab <- table(test_df$presence, test_df$success)
  
  test <- fisher.test(tab)
  
  summary_df <- test_df %>%
    group_by(presence) %>%
    summarise(
      n_total = n(),
      n_success = sum(success == TRUE),
      n_fail = sum(success == FALSE),
      success_rate = round(n_success / n_total * 100, 1),
      fail_rate = round(n_fail / n_total * 100, 1),
      .groups = "drop"
    )
  
  tibble(
    characteristic = var_name,
    n_no = summary_df$n_total[summary_df$presence == "no"],
    n_yes = summary_df$n_total[summary_df$presence == "yes"],
    success_rate_no = summary_df$success_rate[summary_df$presence == "no"],
    success_rate_yes = summary_df$success_rate[summary_df$presence == "yes"],
    fail_rate_no = summary_df$fail_rate[summary_df$presence == "no"],
    fail_rate_yes = summary_df$fail_rate[summary_df$presence == "yes"],
    odds_ratio = unname(test$estimate),
    p_value = test$p.value
  )
}

In [46]:
rq4_exec_has_results <- map_dfr(
  has_vars_exec,
  ~ run_fisher_has_success(.x, df_rq4_exec)
) %>%
  mutate(
    p_value_holm = p.adjust(p_value, method = "holm"),
    significant_holm = p_value_holm < 0.05
  ) %>%
  arrange(p_value_holm)

write_csv(
  rq4_exec_has_results,
  file.path(tables_dir, "rq4_executability_fisher_has_success.csv")
)

rq4_exec_has_results

characteristic,n_no,n_yes,success_rate_no,success_rate_yes,fail_rate_no,fail_rate_yes,odds_ratio,p_value,p_value_holm,significant_holm
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
has_subqueries,2304,696,91.5,74.9,8.5,25.1,0.2754176,6.489809e-28,1.297962e-26,TRUE
has_rollup,2976,24,88.4,0.0,11.6,100.0,0.0000000,7.851378e-23,1.491762e-21,TRUE
has_cte,2292,708,84.8,97.0,15.2,3.0,5.8743812,2.335844e-22,4.204519e-21,TRUE
has_window,2532,468,85.7,98.1,14.3,1.9,8.4772008,4.057595e-18,6.897912e-17,TRUE
has_group_by,324,2676,73.5,89.4,26.5,10.6,3.0419368,8.880823e-14,1.420932e-12,TRUE
has_grouping,2988,12,88.0,0.0,12.0,100.0,0.0000000,1.057422e-11,1.586133e-10,TRUE
has_cube,2988,12,88.0,0.0,12.0,100.0,0.0000000,1.057422e-11,1.586133e-10,TRUE
has_aggregates,324,2676,75.3,89.2,24.7,10.8,2.6963979,7.672697e-11,9.974506e-10,TRUE
has_having,1980,1020,85.3,92.3,14.7,7.7,2.0517728,2.118812e-08,2.542575e-07,TRUE


Rezultatele testelor Fisher exact pentru RQ4 din perspectiva executabilității arată că mai multe caracteristici structurale `has_*` sunt asociate semnificativ cu succesul sau eșecul execuției în scenariile distribuite. După corecția Holm, 13 din cele 20 de caracteristici testate rămân semnificative statistic, ceea ce **susține H14 pe axa executabilității**.

Cele mai problematice caracteristici sunt `has_rollup`, `has_grouping` și `has_cube`, pentru care rata eșec este 100%, lucru deja cunoscut, deoarece clauzele nu sunt executabile în configurații distribuite Citus.

`has_subqueries` este, de asemenea, asociată cu o executabilitate mai slabă: interogările fără subinterogări au o rată de succes de 91,5%, în timp ce cele cu subinterogări au o rată de succes de 74,9%. Odds ratio-ul subunitar (`0,275`) indică o probabilitate mai redusă de succes pentru interogările care conțin subinterogări.

În schimb, unele caracteristici sunt asociate cu rate de succes mai mari. De exemplu, `has_cte` are rată de succes de 97,0% față de 84,8% pentru interogările fără CTE, iar `has_window` are 98,1% față de 85,7%. Similar, `has_group_by`, `has_aggregates`, `has_having`, `has_limit` și `has_order_by` au rate de succes mai mari când caracteristica este prezentă.

Pentru `has_case`, `has_distinct`, `has_full_join`, `has_cross_join`, `has_union` și `has_left_join`, rezultatele nu rămân semnificative după corecția Holm. Prin urmare, pentru aceste caracteristici nu există suficiente dovezi statistice că prezența lor modifică executabilitatea în scenariile distribuite analizate.

În ansamblu, rezultatele arată că executabilitatea în Citus este influențată de anumite caracteristici structurale, dar nu toate construcțiile SQL au același efect.

# Concluzii

În urma testelor inferențiale, **rezultatele susțin respingerea H01 și H04**, **susțin parțial respingerea H03** și **nu oferă suficiente dovezi pentru respingerea generală a H02**.

Pentru RQ1, arhitectura distribuită Citus are un impact semnificativ asupra timpului de execuție și executabilității: reduce timpii pentru interogările finalizate cu succes, dar introduce eșecuri pentru unele query-uri care rulau în baseline.

Pentru RQ2, adăugarea de workeri de la N2 la N5 nu produce un beneficiu semnificativ general. Efectul pozitiv apare doar în anumite configurații, în special la SF50, iar executabilitatea rămâne practic neschimbată.

Pentru RQ3, strategia de distribuire influențează parțial timpul de execuție, în special în unele configurații în care D2 este mai rapidă decât D1. Totuși, diferențele de executabilitate dintre D1 și D2 nu sunt semnificative.

Pentru RQ4, caracteristicile structurale ale interogărilor SQL sunt asociate semnificativ cu performanța și executabilitatea. Cele mai clare limitări apar pentru `ROLLUP`, `CUBE`, `GROUPING` și subinterogări, iar variabilele numerice de complexitate au efecte semnificative, dar în general slabe.

Rezultatele nu sunt interpretate printr-o validare absolută a ipotezelor, ci prin raportare la cele două axe analizate: timpul de execuție și executabilitatea. În cazul în care rezultatele sunt semnificative doar pentru o parte dintre comparații sau doar pentru una dintre axe, ipoteza alternativă este considerată susținută parțial.

Astfel, H01 și H04 sunt respinse pe baza rezultatelor obținute, deoarece testele indică asocieri semnificative atât pentru performanță, cât și pentru executabilitate. H02 nu este respinsă per total, deoarece scalarea de la 2 la 5 workeri nu produce un efect semnificativ general, ci doar punctual. H03 este respinsă parțial, deoarece strategia de distribuire influențează semnificativ timpul în anumite configurații, dar nu modifică semnificativ executabilitatea.